In [ ]:
# How to get your API key

# 1. Open OpenWeatherMap API.
# 2. Create a free account.
# 3. Open your API Keys section.
# 4. Copy your API key and paste it when the Colab code asks.




# ============================================================
# EXPERIMENT NO. 2
# INTERACTING WITH WEB APIs
# Analyzing Weather Data using OpenWeatherMap API
# ============================================================

# STEP 1: Import required libraries

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from getpass import getpass

# STEP 2: Enter your OpenWeatherMap API key
# Get a free API key from https://openweathermap.org/api

API_KEY = getpass("Enter your OpenWeatherMap API key: ")

# STEP 3: Enter the city name

city = input("Enter city name (example: Pune): ")

# STEP 4: Fetch weather forecast data using API

url = "https://api.openweathermap.org/data/2.5/forecast"

params = {
    "q": city,
    "appid": API_KEY,
    "units": "metric"
}

response = requests.get(url, params=params, timeout=30)

# Check whether the API request was successful

if response.status_code != 200:
    print("API Error:", response.status_code)
    print(response.json().get("message", "Unable to retrieve weather data."))

else:

    weather_data = response.json()

    print("\nWeather data successfully retrieved!")
    print("City:", weather_data["city"]["name"])
    print("Country:", weather_data["city"]["country"])

    # STEP 5: Extract relevant weather information

    records = []

    for item in weather_data["list"]:

        record = {
            "Date_Time": item["dt_txt"],
            "Temperature": item["main"]["temp"],
            "Humidity": item["main"]["humidity"],
            "Pressure": item["main"]["pressure"],
            "Wind_Speed": item["wind"]["speed"],
            "Weather": item["weather"][0]["description"],
            "Precipitation": item.get("rain", {}).get("3h", 0)
        }

        records.append(record)

    # STEP 6: Convert the extracted data into a DataFrame

    df = pd.DataFrame(records)

    print("\n========== WEATHER DATA ==========")
    print(df.head())

    # STEP 7: Explore the dataset

    print("\nDataset Shape:")
    print(df.shape)

    print("\nDataset Information:")
    df.info()

    print("\nStatistical Description:")
    print(df.describe())

    # STEP 8: Convert Date_Time into datetime format

    df["Date_Time"] = pd.to_datetime(df["Date_Time"])

    # STEP 9: Check and handle missing values

    print("\nMissing Values:")
    print(df.isnull().sum())

    # Fill missing numerical values with their column mean

    numeric_columns = [
        "Temperature", "Humidity", "Pressure",
        "Wind_Speed", "Precipitation"
    ]

    for column in numeric_columns:
        df[column] = df[column].fillna(df[column].mean())

    # Remove duplicate records

    df = df.drop_duplicates()

    print("\nDataset after cleaning:")
    print(df.head())

    # STEP 10: Perform weather data analysis

    print("\n========== WEATHER ANALYSIS ==========")

    print("Average Temperature:",
          round(df["Temperature"].mean(), 2), "°C")

    print("Maximum Temperature:",
          round(df["Temperature"].max(), 2), "°C")

    print("Minimum Temperature:",
          round(df["Temperature"].min(), 2), "°C")

    print("Average Humidity:",
          round(df["Humidity"].mean(), 2), "%")

    print("Average Wind Speed:",
          round(df["Wind_Speed"].mean(), 2), "m/s")

    print("Average Pressure:",
          round(df["Pressure"].mean(), 2), "hPa")

    print("Total Forecast Precipitation:",
          round(df["Precipitation"].sum(), 2), "mm")

    # STEP 11: Aggregate data by date

    df["Date"] = df["Date_Time"].dt.date

    daily_weather = df.groupby("Date").agg({
        "Temperature": "mean",
        "Humidity": "mean",
        "Wind_Speed": "mean",
        "Pressure": "mean",
        "Precipitation": "sum"
    }).reset_index()

    print("\n========== DAILY WEATHER SUMMARY ==========")
    print(daily_weather)

    # STEP 12: Visualize temperature changes over time

    plt.figure(figsize=(12, 5))

    plt.plot(
        df["Date_Time"],
        df["Temperature"],
        marker="o"
    )

    plt.title("Temperature Changes Over Time")
    plt.xlabel("Date and Time")
    plt.ylabel("Temperature (°C)")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # STEP 13: Visualize humidity changes

    plt.figure(figsize=(12, 5))

    plt.plot(
        df["Date_Time"],
        df["Humidity"],
        marker="o"
    )

    plt.title("Humidity Changes Over Time")
    plt.xlabel("Date and Time")
    plt.ylabel("Humidity (%)")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # STEP 14: Visualize daily average temperature

    plt.figure(figsize=(10, 5))

    plt.bar(
        daily_weather["Date"].astype(str),
        daily_weather["Temperature"]
    )

    plt.title("Daily Average Temperature")
    plt.xlabel("Date")
    plt.ylabel("Temperature (°C)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # STEP 15: Visualize wind speed

    plt.figure(figsize=(12, 5))

    plt.plot(
        df["Date_Time"],
        df["Wind_Speed"],
        marker="o"
    )

    plt.title("Wind Speed Variation")
    plt.xlabel("Date and Time")
    plt.ylabel("Wind Speed (m/s)")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # STEP 16: Visualize relationship between
    # temperature and humidity

    plt.figure(figsize=(8, 5))

    plt.scatter(
        df["Temperature"],
        df["Humidity"]
    )

    plt.title("Temperature vs Humidity")
    plt.xlabel("Temperature (°C)")
    plt.ylabel("Humidity (%)")
    plt.grid(True)
    plt.show()

    # STEP 17: Create a correlation heatmap

    plt.figure(figsize=(8, 5))

    correlation = df[numeric_columns].corr()

    sns.heatmap(
        correlation,
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )

    plt.title("Weather Parameters Correlation")
    plt.tight_layout()
    plt.show()

    # STEP 18: Save weather data to CSV

    df.to_csv("weather_data.csv", index=False)

    daily_weather.to_csv(
        "daily_weather_summary.csv",
        index=False
    )

    print("\nExperiment completed successfully!")

    print("Weather data saved as weather_data.csv")
    print("Daily summary saved as daily_weather_summary.csv")

    # Download the weather dataset

    from google.colab import files
    files.download("weather_data.csv")
